<span style='font-size:50px; font-weight:bold;'>MOVIES DATA SCRAPPER</span>

---

In [1]:
# import libraries
import time
import random
import json
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.options import Options

from bs4 import BeautifulSoup

# Setup Driver:

<span style='font-size:18px;'>
    ➤ Setting up the selenium driver to open IMDB pages
</span>

In [2]:
def setup_driver():
    options = Options()
    
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-infobars")
    options.add_argument("--disable-extensions")
    
    # OPTIONAL (run in background)
    # options.add_argument("--headless")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    return driver

# JSON Extraction:

<span style='font-size:18px;'>
    ➤ Defining function to extract the json file from the IMDB page
</span>

In [3]:
def extract_json_ld(soup):
    scripts = soup.select('script[type="application/ld+json"]')
    
    for sc in scripts:
        try:
            data = json.loads(sc.string)
            if "aggregateRating" in data:
                return data
        except:
            continue
    
    return None

# Convert duration in minutes:

<span style='font-size:18px;'> ➤ In JSON file, duration is in ISO fomrat `PT2H32M`, so converting it into minutes
</span>

In [4]:
def convert_iso_duration(duration):
    hours = re.search(r'(\d+)H', duration)
    minutes = re.search(r'(\d+)M', duration)

    total = 0
    if hours:
        total += int(hours.group(1)) * 60
    if minutes:
        total += int(minutes.group(1))

    return total if total > 0 else None

# Movie Details Extraction:

<span style='font-size:18px;'>➤ Defining function to extract all the movies details from the page <br>
    ➤ Movies data are extracted using `beautiful soup`
</span>

In [5]:
import pprint

def scrape_movie(driver, url):

    try:
        driver.get(url)

        # WAIT like human
        time.sleep(random.uniform(3, 5))

        soup = BeautifulSoup(driver.page_source, "html.parser")

        json_data = extract_json_ld(soup)
        if not json_data:
            print("JSON missing:", url)
            return None

        # initializing features to make sure the row doesn't shift if value is unable to scrape
        data = {
            'title': None,
            'genres': None,
            'description': None,
            'keywords': None,
            'release_date': None,
            'duration': None,
            'content_rating': None,
            'director': None,
            'star_1': None,
            'star_2': None,
            'star_3': None,
            'countries': None,
            'languages': None,
            'rating': None,
            'metascore': None,
            'votes': None,
            'user_reviews': None,
            'critic_reviews': None,
            'watchlist_count': None,
            'gross': None,
            'budget': None
        }

        # BASIC INFO
        data['title'] = json_data.get('name')
        data['genres'] = "|".join(json_data.get('genre', []))
        data['release_date'] = json_data.get('datePublished')

        runtime_iso = json_data.get('duration')
        data['duration'] = convert_iso_duration(runtime_iso)

        rating = json_data.get('aggregateRating', {})
        data['rating'] = rating.get('ratingValue')
        data['votes'] = rating.get('ratingCount')

        data['content_rating'] = json_data.get('contentRating')
        data['description'] = json_data.get('description')
        data['keywords'] = json_data.get('keywords')

        # ACTORS
        actors = json_data.get('actor', [])
        data['star_1'] = actors[0]['name'] if len(actors) > 0 else None
        data['star_2'] = actors[1]['name'] if len(actors) > 1 else None
        data['star_3'] = actors[2]['name'] if len(actors) > 2 else None

        # DIRECTOR
        directors = json_data.get('director', [])
        data['director'] = directors[0]['name'] if directors else None

        # Countries
        try:
            country_block = soup.select_one('li[data-testid="title-details-origin"]')
            
            if country_block:
                countries = country_block.select('a')
                data['countries'] = '|'.join([c.text.strip() for c in countries])
        except:
            data['countries'] = None

         # Languages
        try:
            language_block = soup.select_one('li[data-testid="title-details-languages"]')
            
            if language_block:
                languages = language_block.select('a')
                data['languages'] = '|'.join([l.text.strip() for l in languages])
        except:
            data['languages'] = None

         # Watchlist Count
        try:
            watchlist = soup.select_one('div[data-testid="tm-box-wl-count"]')
            if watchlist:
                text = watchlist.text.strip()
                # Example: "Added by 646K users"
                data['watchlist_count'] = text.replace("Added by", "").replace("users", "").strip()
        except:
            data['watchlist_count'] = None

        # 🧾 Reviews Section (User, Critic, Metascore)
        try:
            review_blocks = soup.select('ul[data-testid="reviewContent-all-reviews"] li')
            
            for block in review_blocks:
                label = block.select_one('.label')
                score = block.select_one('.score')
                
                if label and score:
                    label_text = label.text.strip().lower()
                    score_text = score.text.strip()
                    
                    if "user reviews" in label_text:
                        data['user_reviews'] = score_text
                    
                    elif "critic reviews" in label_text:
                        data['critic_reviews'] = score_text
                    
                    elif "metascore" in label_text:
                        data['metascore'] = score_text
        
        except:
            pass
            

        # FINANCIALS
        try:
            data['gross'] = soup.select_one('[data-testid="title-boxoffice-cumulativeworldwidegross"] div').text.strip()
        except:
            data['gross'] = None

        try:
            data['budget'] = soup.select_one('[data-testid="title-boxoffice-budget"] div').text.strip()
        except:
            data['budget'] = None

        try:
            block = soup.find("li", {"data-testid": "title-boxoffice-openingweekenddomestic"})
            data['opening_weekend'] = block.find("span", class_="ipc-metadata-list-item__list-content-item").text.strip()
        except:
            data['opening_weekend'] = None

        # print('\n')
        # pprint.pprint(data, indent=4)
        return data

    except Exception as e:
        print("Error:", e, url)
        return None

# Combining Movie Links:

<span style='font-size:18px;'>➤ Combining all the movie links extracted from different listing pages <br>
    ➤ `glob` is used to get the all .csv file fo different listing pages and combined it into one DataFrame
</span>

In [6]:
import glob

files = glob.glob("raw_data/*.csv")

df_links = pd.concat([pd.read_csv(f) for f in files])
print("Total movies: ", len(df_links))

Total movies:  7005


In [7]:
mul_cat = df_links.groupby('movie_url')['Scrape_Category'].count()
cat_df = pd.DataFrame(mul_cat).reset_index()
cat_df

,movie_url,Scrape_Category
0,https://www.imdb.com/search/title/,5
1,https://www.imdb.com/title/tt0035423/,1
2,https://www.imdb.com/title/tt0076276/,2
3,https://www.imdb.com/title/tt0078935/,1
4,https://www.imdb.com/title/tt0079285/,1
...,...,...
5645,https://www.imdb.com/title/tt9853500/,1
5646,https://www.imdb.com/title/tt9866072/,1
5647,https://www.imdb.com/title/tt9881586/,1
5648,https://www.imdb.com/title/tt9893250/,2


In [8]:
df_links = df_links.groupby('movie_url').agg({
    'Scrape_Category': lambda x: '|'.join(set(x))
}).reset_index()

In [9]:
df_links

,movie_url,Scrape_Category
0,https://www.imdb.com/search/title/,Most Popular|Top Rated|Old Movies|Mid Rated|Lo...
1,https://www.imdb.com/title/tt0035423/,Mid Rated
2,https://www.imdb.com/title/tt0076276/,Old Movies|Top Rated
3,https://www.imdb.com/title/tt0078935/,Old Movies
4,https://www.imdb.com/title/tt0079285/,Old Movies
...,...,...
5645,https://www.imdb.com/title/tt9853500/,Most Popular
5646,https://www.imdb.com/title/tt9866072/,Mid Rated
5647,https://www.imdb.com/title/tt9881586/,Top Rated
5648,https://www.imdb.com/title/tt9893250/,Mid Rated|Most Popular


In [10]:
df_links = df_links.iloc[1:].reset_index(drop=True)

In [11]:
df_links

,movie_url,Scrape_Category
0,https://www.imdb.com/title/tt0035423/,Mid Rated
1,https://www.imdb.com/title/tt0076276/,Old Movies|Top Rated
2,https://www.imdb.com/title/tt0078935/,Old Movies
3,https://www.imdb.com/title/tt0079285/,Old Movies
4,https://www.imdb.com/title/tt0079579/,Old Movies
...,...,...
5644,https://www.imdb.com/title/tt9853500/,Most Popular
5645,https://www.imdb.com/title/tt9866072/,Mid Rated
5646,https://www.imdb.com/title/tt9881586/,Top Rated
5647,https://www.imdb.com/title/tt9893250/,Mid Rated|Most Popular


In [6]:
#for new movies (model testing)
df_new = pd.read_csv('raw_data/New_Movies.csv')
df_new.head()

,movie_url,Scrape_Category
0,https://www.imdb.com/title/tt28459771/,New Movies
1,https://www.imdb.com/title/tt34896285/,New Movies
2,https://www.imdb.com/title/tt37140876/,New Movies
3,https://www.imdb.com/title/tt37803534/,New Movies
4,https://www.imdb.com/title/tt39875252/,New Movies


# Batch Scrapper:

<span style='font-size:18px;'>➤ Instead of scraping all movies at once, a batch-wise apprroach is used to avoid blocking from IMDB servers. <br>
    ➤ Total of `300 movies` scraped in one batch.
</span>

In [7]:
import os
from datetime import datetime

def run_batch_scraper(batch_size=300):

    # df = df_new
    df = df_links

    output_file = "data/Imdb_movies_data.csv"

    # Resume logic
    if os.path.exists(output_file):
        existing_df = pd.read_csv(output_file)
        start_index = len(existing_df)
        print(f"Resuming from index {start_index}")
    else:
        start_index = 0
        print("Starting fresh scraping")

    end_index = min(start_index + batch_size, len(df))

    driver = setup_driver()
    print("Batch Starting at: ",datetime.now())

    for i in range(start_index, end_index):

        url = df.loc[i, "movie_url"]
        category = df.loc[i,"Scrape_Category"]
        print(f"Scraping {i+1}/{len(df)}")

        movie = scrape_movie(driver, url)

        if movie:
            movie['movie_url'] = url
            movie['category'] = category

            pd.DataFrame([movie]).to_csv(
                output_file,
                mode='a',
                header=not os.path.exists(output_file),
                index=False
            )

        time.sleep(random.uniform(2,4))

    driver.quit()

    print(f"Batch completed at {datetime.now()}: {start_index} → {end_index}")

In [12]:
# Running the scrapper
run_batch_scraper(batch_size=300)

Resuming from index 1196
Batch Starting at:  2026-04-03 18:57:30.008461
Scraping 1197/1301
Scraping 1198/1301
Scraping 1199/1301
Scraping 1200/1301
Scraping 1201/1301
Scraping 1202/1301
Scraping 1203/1301
Scraping 1204/1301
Scraping 1205/1301
Scraping 1206/1301
Scraping 1207/1301
Scraping 1208/1301
Scraping 1209/1301
Scraping 1210/1301
Scraping 1211/1301
Scraping 1212/1301
Scraping 1213/1301
Scraping 1214/1301
Scraping 1215/1301
Scraping 1216/1301
Scraping 1217/1301
Scraping 1218/1301
Scraping 1219/1301
Scraping 1220/1301
Scraping 1221/1301
Scraping 1222/1301
Scraping 1223/1301
Scraping 1224/1301
Scraping 1225/1301
Scraping 1226/1301
Scraping 1227/1301
Scraping 1228/1301
Scraping 1229/1301
Scraping 1230/1301
Scraping 1231/1301
Scraping 1232/1301
Scraping 1233/1301
Scraping 1234/1301
Scraping 1235/1301
Scraping 1236/1301
Scraping 1237/1301
Scraping 1238/1301
Scraping 1239/1301
Scraping 1240/1301
Scraping 1241/1301
Scraping 1242/1301
Scraping 1243/1301
Scraping 1244/1301
Scraping 1245/13

---